# Submissão 2 — Modelo B (DistilRoBERTa)

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from src.transformer_utils import InferenceDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

ID2LABEL_OUT = {
    0: 'Google',
    1: 'Anthropic',
    2: 'Meta',
    3: 'OpenAI',
    4: 'Human'
}

## *carregar dados e modelo*

In [ ]:
df_subm = pd.read_csv('subm2.csv', sep=';')

texts = df_subm['Text'].fillna('').tolist()

print(f'amostras a classificar: {len(texts)}')
print(df_subm.head(3))

In [ ]:
MODEL_PATH = '../models/model_bert'
MAX_LEN    = 128

tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)
model     = RobertaForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()
print('modelo carregado!')

In [ ]:
subm_ds     = InferenceDataset(texts, tokenizer, MAX_LEN)
subm_loader = DataLoader(subm_ds, batch_size=32)
print(f'batches: {len(subm_loader)}')

## *inferência*

In [ ]:
preds_all = []

with torch.no_grad():
    for batch in subm_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=1).cpu().tolist()
        preds_all.extend(preds)

df_subm['Label'] = [ID2LABEL_OUT[p] for p in preds_all]
print(df_subm['Label'].value_counts())

In [ ]:
filename = 'subm2-g2-MEI-B.csv'
df_subm[['ID', 'Label']].to_csv(filename, index=False, sep=';')
print(f"Ficheiro '{filename}' guardado com sucesso!")
print(df_subm[['ID', 'Label']].head())